# tkh-phase-2-project - Financial Fraud Detection - Model notebook

In [1]:
import datetime
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.model_selection import train_test_split, RandomizedSearchCV, GridSearchCV
import time

## Load dataset

Load the transformed dataset.

In [2]:
df = pd.read_csv('../data/transform_data.csv')

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6362620 entries, 0 to 6362619
Data columns (total 8 columns):
 #   Column            Dtype  
---  ------            -----  
 0   amount            float64
 1   new_balance_orig  float64
 2   new_balance_dest  float64
 3   is_fraud          int64  
 4   cash_out          bool   
 5   debit             bool   
 6   payment           bool   
 7   transfer          bool   
dtypes: bool(4), float64(3), int64(1)
memory usage: 218.4 MB


## Prepare predictors and target variables

Separate the `is_fraud` column from our predictor columns.

In [4]:
X = df.drop('is_fraud', axis=1)
y = df['is_fraud']
print('X.shape', X.shape)
print('y.shape', y.shape)

X.shape (6362620, 7)
y.shape (6362620,)


## Split our data into train/test sets

Use 70% training and 30% testing split. Set a random state to ensure reproducibility.

In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
print('X_train.shape', X_train.shape)
print('y_train.shape', y_train.shape)
print('X_test.shape', X_test.shape)
print('y_test.shape', y_test.shape)

X_train.shape (4453834, 7)
y_train.shape (4453834,)
X_test.shape (1908786, 7)
y_test.shape (1908786,)


## First model - default hyperparameters

First, initialize and fit a Random Forest Classifier with the default hyperparameters. In to know how long to this fitting takes for future runs, we record the time it takes to complete. Second, we use our model to predict our test dataset. And lastly, we print our confusion matrix and classification report in order to evaluate our model.

In [16]:
print('Begin fitting a Random Forest model')
print('This will take a long time')
print('Start time:', datetime.datetime.now())
start = time.time()

rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)
elapsed_time = time.time() - start

print(f'Elapsed time: {np.floor(elapsed_time / 60):.0f} min, {round(elapsed_time % 60)} sec')

Begin fitting a Random Forest model
This will take a long time
Start time: 2026-04-18 17:30:12.660768
Elapsed time: 11 min, 50 sec


In [17]:
print('Begin predicting our test data using our first model')
print("This won't take long")
start = time.time()

yhat = rf.predict(X_test)

elapsed_time = time.time() - start
print(f'Elapsed time: {np.floor(elapsed_time / 60):.0f} min, {round(elapsed_time % 60)} sec')

Begin predicting our test data using our first model
This won't take long
Elapsed time: 0 min, 11 sec


In [18]:
confusion = confusion_matrix(y_test, yhat)
class_report = classification_report(y_test, yhat)

print("Confusion Matrix \n", confusion)
print("\nClassification Report\n", class_report)

Confusion Matrix 
 [[1905922     429]
 [   1061    1374]]

Classification Report
               precision    recall  f1-score   support

           0       1.00      1.00      1.00   1906351
           1       0.76      0.56      0.65      2435

    accuracy                           1.00   1908786
   macro avg       0.88      0.78      0.82   1908786
weighted avg       1.00      1.00      1.00   1908786



## Second model - perform a hyperparameter search

By tuning its hyperparameteres, we can be build a Random Forest model that performs even better than our previous one. To find the hyperparameter values that arguably work the best, we do a randomized search on our hyperparameteres with cross validation. For our search, we use 5 folds and tweak the following four hyperparameters, whose descriptions come from [this scikit-learn doc page][sklearn-doc-page]. The range of values for these hyperparameters was borrowed from our Random Forest code-along lab.

    1. criterion - "The function to measure the quality of a split."
    2. max_depth - "The maximum depth of the tree."
    3. max_features - "The number of features to consider when looking for the best split"
    4. min_samples_split - "The minimum number of samples required to be at a leaf node."

As part of our search, we generate many models for comparison, so this procedure will take much longer than our previous step where we just built one model. I ran this step overnight and it took me almost 7.5 hours on my aging desktop, which has an 6-core processor from 2018 (Intel Core i5-8500 @ 4.10 GHz).

After fitting our crossfold model with our training data, we inspect the hyperparameters that worked best and predict our test data using those hyperparameters. Finally, we print our confusion matrix and classification report, so that we evaluate our model.

[sklearn-doc-page]: https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html

In [6]:
rf_randomized_cv = RandomizedSearchCV(
    cv=5,
    estimator=RandomForestClassifier(random_state=42),
    param_distributions={
        'criterion': ['gini', 'entropy','log_loss'],
        'max_depth': range(10, 100, 20),
        'max_features': ['sqrt', 'log2'],
        'min_samples_split': range(2, 50, 10)
    },
    random_state=42
)

In [7]:
print('Begin fitting a Random Forest with cross validation')
print('This will take a very, very long time')
print('Start time:', datetime.datetime.now())
start = time.time()

rf_randomized_cv.fit(X_train, y_train)
elapsed_time = time.time() - start

print(f'Elapsed time: {np.floor(elapsed_time / 3600):.0f} hr, {(elapsed_time / 60) % 60:.0f} min, {round(elapsed_time % 60)} sec')

Begin fitting a Random Forest with cross validation
This will take a very, very long time
Start time: 2026-04-17 22:10:08.958299
Elapsed time: 7 hr, 23 min, 57 sec


In [8]:
rf_best = rf_randomized_cv.best_params_
print('Best parameters from our hyperparameter search\n', rf_best)

Best parameters from our hyperparameter search
 {'min_samples_split': 42, 'max_features': 'sqrt', 'max_depth': 30, 'criterion': 'entropy'}


In [10]:
print('Begin predicting our test data using the best model hyperparameters')
print("This won't take long")
start = time.time()

yhat = rf_randomized_cv.best_estimator_.predict(X_test)

elapsed_time = time.time() - start
print(f'Elapsed time: {np.floor(elapsed_time / 60):.0f} min, {round(elapsed_time % 60)} sec')

Begin predicting our test data using the best model hyperparameters
This won't take long
Elapsed time: 0 min, 9 sec


In [11]:
confusion = confusion_matrix(y_test, yhat)
class_report = classification_report(y_test, yhat)

print("Confusion Matrix \n", confusion)
print("\nClassification Report\n", class_report)

Confusion Matrix 
 [[1906039     312]
 [    864    1571]]

Classification Report
               precision    recall  f1-score   support

           0       1.00      1.00      1.00   1906351
           1       0.83      0.65      0.73      2435

    accuracy                           1.00   1908786
   macro avg       0.92      0.82      0.86   1908786
weighted avg       1.00      1.00      1.00   1908786



Above, our confusion matrix shows that we had 1,906,039 true positives, 1,571 true negatives, 312 false positives, and 864 false negatives. We know that true positives mean transactions correctly classified as non-fraud, true negatives mean transactions correctly classified as fraud, false positives mean transactions incorrectly classified as non-fraud, and false negatives mean transactions incorrectly classified as fraud.

Below is a lecture slide from our class that shows how to interpret a confusion matrix.

<img alt="Confusion matrix graphic from lecture slides" src="../docs/confusion_matrix_slide.png" width=500 />

In the classification report above, we see that our `f1-score` for the positive class (a fraudulent transaction) improved from 0.65, in our previous model, to 0.73 here, which is an appreciable difference. 

Recall that f1-score = 2 * (Precision * Recall) / (Precision + Recall), and that the higher the the f1-score the more balanced the model's Precision and Recall are. Precision is defined as of the positive predictions, how many were actually correct. Recall is defined as the of the actual positives, how many were predicted correctly.

## Third model - use what we learned from our hyperparameter search

In this third model, we initialize, fit, and predict a model using the hyperparameters we determined to work best in our previous step. This work is probably unnecessary because in the previous step we examined a model with those same hyperparameters when we accessed `best_estimator_` attribute of our search object. The assignment instructions were worded in a way that sounded like this third model was required, so I'm including it here.

In [13]:
print('Begin fitting a Random Forest model')
print('This will take a long time')
print('Start time:', datetime.datetime.now())
start = time.time()

# TODO fill in hyperparameters
rf = RandomForestClassifier(
    criterion='entropy',
    max_depth=30,
    max_features='sqrt',
    min_samples_split=42,
    random_state=42,
)
rf.fit(X_train, y_train)
elapsed_time = time.time() - start

print(f'Elapsed time: {np.floor(elapsed_time / 60):.0f} min, {round(elapsed_time % 60)} sec')

Begin fitting a Random Forest model
This will take a long time
Start time: 2026-04-18 13:17:56.910196
Elapsed time: 11 min, 30 sec


In [14]:
print('Begin predicting our test data using the best model hyperparameters')
print("This won't take long")
start = time.time()

yhat = rf.predict(X_test)

elapsed_time = time.time() - start
print(f'Elapsed time: {np.floor(elapsed_time / 60):.0f} min, {round(elapsed_time % 60)} sec')

Begin predicting our test data using the best model hyperparameters
This won't take long
Elapsed time: 0 min, 9 sec


In [15]:
confusion = confusion_matrix(y_test, yhat)
class_report = classification_report(y_test, yhat)

print("Confusion Matrix \n", confusion)
print("\nClassification Report\n", class_report)

Confusion Matrix 
 [[1906039     312]
 [    864    1571]]

Classification Report
               precision    recall  f1-score   support

           0       1.00      1.00      1.00   1906351
           1       0.83      0.65      0.73      2435

    accuracy                           1.00   1908786
   macro avg       0.92      0.82      0.86   1908786
weighted avg       1.00      1.00      1.00   1908786



Since the above model and the second model are identical, the confusion matrix and classification reports are the same.